# 🏢 Week 4 Assignment — Company Database: Joins, Errors & Debugging

**Course:** Python Programming — Five-Week Foundations  
**Week:** 4 — Multi-table schema · INNER JOIN · LEFT JOIN · GROUP BY · Error handling · Debugging  
**Estimated time:** 3–4 hours  
**Total points:** 100  

---

## The Database

This week you work with `company_w4.db` — a four-table database that models a real company structure.

```
departments              employees                 projects              assignments
────────────────         ─────────────────         ───────────────       ──────────────────
id  PK                   id  PK                    id  PK                id  PK
name                     name                      name                  employee_id → employees
budget                   age                       dept_id  → depts      project_id  → projects
location                 dept_id  → depts          budget                role
                         city                      status                hours
                         salary                    start_date
                         start_date                end_date
                         email
```

**Key facts built into the data:**
- `Legal` department has **no employees** — useful for LEFT JOIN practice
- `Leo Nakamura` and `Mia Torres` are contractors with **no department** (`dept_id = NULL`)
- `Laura Chen`, `Wendy Park`, and `Zoe Williams` have **no project assignments**
- `Skunkworks AI` project has **no sponsoring department** (`dept_id = NULL`)

**Files needed in this folder:**
| File | Description |
|------|-------------|
| `week4_assignment.ipynb` | This notebook |
| `company_w4.db` | The four-table database |
| `debug_me.py` | Buggy script for the debugging exercise |

> ⚠️ **Before you start:** activate your virtual environment — `source venv/bin/activate`

---

## Reference — JOIN Types

A JOIN combines rows from two tables based on a matching column. The type of JOIN determines what happens when there is **no match**.

### INNER JOIN — only matched rows

Returns a row only when both tables have a matching value. Unmatched rows from either table are **excluded**.

```python
# Only employees who have a department
cursor.execute("""
    SELECT e.name, d.name AS department
    FROM   employees e
    INNER JOIN departments d ON d.id = e.dept_id
""")
# Result: 25 rows — Leo and Mia (NULL dept_id) are excluded
```

### LEFT JOIN — all rows from the left table

Returns **every row from the left table**. If no match exists in the right table, the right-side columns are `NULL`.

```python
# All employees, including those with no department
cursor.execute("""
    SELECT e.name, d.name AS department
    FROM   employees e
    LEFT JOIN departments d ON d.id = e.dept_id
""")
# Result: 27 rows — Leo and Mia appear with department = None
```

You can also flip it to find departments with no employees:

```python
# All departments, including those with no employees
cursor.execute("""
    SELECT d.name AS department, e.name AS employee
    FROM   departments d
    LEFT JOIN employees e ON e.dept_id = d.id
""")
# Legal appears with employee = None
```

### Chaining multiple JOINs

You can join more than two tables by chaining JOIN clauses:

```python
cursor.execute("""
    SELECT  e.name        AS employee,
            d.name        AS department,
            p.name        AS project,
            a.role,
            a.hours
    FROM    assignments a
    INNER JOIN employees   e ON e.id = a.employee_id
    INNER JOIN projects    p ON p.id = a.project_id
    LEFT  JOIN departments d ON d.id = e.dept_id
""")
# One row per assignment, showing the employee, their dept, and the project
```

### COUNT(\*) vs COUNT(column) with LEFT JOIN

When you use LEFT JOIN with GROUP BY, `COUNT(*)` gives the wrong result for empty groups:

```python
# ❌ COUNT(*) counts the NULL placeholder row — Legal shows 1, not 0
SELECT d.name, COUNT(*) AS headcount
FROM departments d
LEFT JOIN employees e ON e.dept_id = d.id
GROUP BY d.id

# ✅ COUNT(e.id) ignores NULLs — Legal correctly shows 0
SELECT d.name, COUNT(e.id) AS headcount
FROM departments d
LEFT JOIN employees e ON e.dept_id = d.id
GROUP BY d.id
```

| JOIN type | Left table rows | Right table rows | When no match |
|-----------|----------------|-----------------|---------------|
| `INNER JOIN` | Only matched | Only matched | Row excluded |
| `LEFT JOIN` | **All** | Only matched | Right columns = `NULL` |

---

## Reference — Error Handling with `try` / `except`

You used `try`/`except` in Week 2 for CSV parsing. This week we apply the same pattern to database operations, which introduces new error types.

### Common database errors

| Error | When it happens |
|-------|-----------------|
| `sqlite3.OperationalError` | Table does not exist, bad SQL syntax, file not found |
| `sqlite3.IntegrityError` | Violates a constraint (e.g. inserting a duplicate UNIQUE value) |
| `sqlite3.Error` | Catches any sqlite3 error — use as a fallback |

```python
import sqlite3

def safe_insert(name, salary):
    try:
        conn = sqlite3.connect("company_w4.db")
        conn.row_factory = sqlite3.Row
        cursor = conn.cursor()
        cursor.execute(
            "INSERT INTO employees (name, city, salary) VALUES (?, ?, ?)",
            (name, "Remote", salary)
        )
        conn.commit()
        print(f"Inserted: {name}")
    except sqlite3.IntegrityError as e:
        print(f"Constraint violation: {e}")
    except sqlite3.OperationalError as e:
        print(f"Database error: {e}")
    except Exception as e:
        print(f"Unexpected error: {e}")
    finally:
        conn.close()   # always runs, even if an error occurred
```

### The `finally` block

`finally` runs **no matter what** — whether the code succeeded or raised an error. It is the right place to close a database connection:

```python
try:
    conn = sqlite3.connect("company_w4.db")
    # ... do work ...
except sqlite3.Error as e:
    print(f"Error: {e}")
finally:
    conn.close()   # guaranteed to run
```

---

## Reference — Reading Tracebacks

When Python raises an error, it prints a **traceback** — a trail showing exactly where things went wrong. Read it **from the bottom up**:

```
Traceback (most recent call last):          ← ignore this line
  File "debug_me.py", line 211, in main     ← where it was called from
    depts = get_all_departments()           ← the call
  File "debug_me.py", line 38, in get_all_departments
    return dept                             ← the actual problem line
NameError: name 'dept' is not defined       ← error type and message  ← START HERE
```

**Reading strategy:**
1. Look at the **last line** — this is the error type and message
2. Look at the **second-to-last block** — this is the exact file, line number, and code that failed
3. Read upward to understand the call chain that led there

### Common errors you'll see this week

| Error | Typical cause |
|-------|---------------|
| `NameError: name 'x' is not defined` | Typo in variable name |
| `KeyError: 'column_name'` | Column not included in SELECT |
| `TypeError: function takes at most 2 arguments` | Passed a string instead of a tuple to `cursor.execute` |
| `sqlite3.OperationalError: no such table` | DB file not in the same folder as the script |
| `ZeroDivisionError: division by zero` | Dividing by a variable that can be zero |

---

## Task 1 — Explore the Schema
**10 points**

Before writing queries, get familiar with the four-table structure.

**Requirements:**
1. Connect to `company_w4.db` and set `row_factory = sqlite3.Row`.
2. List all four table names using `SELECT name FROM sqlite_master WHERE type='table'`.
3. For **each** table, print its column names using `PRAGMA table_info(tablename)`.
4. Print the row count for each table.
5. Wrap the entire block in `try` / `except sqlite3.OperationalError` so a helpful message prints if the database file is missing.

> 💡 **Hint:** `PRAGMA table_info(employees)` returns one row per column. Each row has a `name` field and a `type` field.

In [8]:
# Task 1 — Explore the schema
import sqlite3

try:
    conn = sqlite3.connect("company_w4.db")
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    # 2. List all tables
    print("--- Tables ---")
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
    tables = [row["name"] for row in cursor.fetchall()]
    for t in tables:
        print(f"  {t}")

    # 3 & 4. For each table, print columns and row count
    for table in tables:
        cursor.execute(f"PRAGMA table_info({table})")
        cols = [(r["name"], r["type"]) for r in cursor.fetchall()]
        cursor.execute(f"SELECT COUNT(*) AS n FROM {table}")
        count = cursor.fetchone()["n"]
        print(f"\n{table} ({count} rows)")
        for col_name, col_type in cols:
            print(f"  {col_name:<15} {col_type}")

    conn.close()

except sqlite3.OperationalError as e:
    print(f"Database error: {e}")
    print("Make sure company_w4.db is in the same folder as this notebook.")

--- Tables ---
  assignments
  departments
  employees
  projects
  sqlite_sequence

assignments (30 rows)
  id              INTEGER
  employee_id     INTEGER
  project_id      INTEGER
  role            TEXT
  hours           INTEGER

departments (5 rows)
  id              INTEGER
  name            TEXT
  budget          REAL
  location        TEXT

employees (27 rows)
  id              INTEGER
  name            TEXT
  age             INTEGER
  dept_id         INTEGER
  city            TEXT
  salary          REAL
  start_date      TEXT
  email           TEXT

projects (18 rows)
  id              INTEGER
  name            TEXT
  dept_id         INTEGER
  budget          REAL
  status          TEXT
  start_date      TEXT
  end_date        TEXT

sqlite_sequence (4 rows)
  name            
  seq             


---
## Task 2 — INNER JOIN Queries
**15 points**

INNER JOIN returns only rows that have a match in **both** tables.

**Requirements:**
1. Query every employee with their department name. Print `name`, `department`, `city`, and `salary`. Count the results — notice it is **25**, not 27 (the two contractors with no dept are excluded).
2. Query all project assignments showing `employee name`, `project name`, and `role`. Use INNER JOINs across `assignments`, `employees`, and `projects`. Order by employee name.
3. Find all employees assigned to the **`Data Lake Migration`** project — show their name, role, and hours. Use a `WHERE` clause after the JOIN.
4. Count the number of assignments per employee, ordered by most assignments first. Show only employees with **2 or more** assignments using `HAVING COUNT(*) >= 2`.

> 💡 **Hint for requirement 4:** `HAVING` filters *after* grouping, the same way `WHERE` filters individual rows before grouping. You cannot use `WHERE` on an aggregate like `COUNT(*)`.

In [9]:
# Task 2 — INNER JOIN
conn = sqlite3.connect("company_w4.db")
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

# 1. Employees with department name
print("--- Employees with Department (INNER JOIN) ---")
cursor.execute("""SELECT e.name AS employee, d.name AS department
                    FROM employees e
                    INNER JOIN departments d ON e.dept_id = d.id
                    ORDER BY employee""")
for row in cursor.fetchall():
    print(f"  {row['employee']:<20} {row['department']}")

# 2. All assignments — employee, project, role
print("\n--- All Project Assignments ---")
cursor.execute("""SELECT e.name AS employee, p.name AS project, a.role
                    FROM assignments a
                    INNER JOIN employees e ON a.employee_id = e.id
                    INNER JOIN projects p ON a.project_id = p.id
                    ORDER BY employee""")
for row in cursor.fetchall():
    print(f"  {row['employee']:<20} {row['project']:<25} {row['role']}")

# 3. Who is on Data Lake Migration?
print("\n--- Data Lake Migration Team ---")
cursor.execute("""SELECT e.name AS employee, a.role
                    FROM assignments a
                    INNER JOIN employees e ON a.employee_id = e.id
                    INNER JOIN projects p ON a.project_id = p.id
                    WHERE p.name = 'Data Lake Migration'
                    ORDER BY employee""")
for row in cursor.fetchall():
    print(f"  {row['employee']:<20} {row['role']}")

# 4. Employees with 2+ assignments
print("\n--- Employees with 2+ Assignments ---")
cursor.execute("""SELECT e.name as employee, a.count
                    FROM employees e
                    INNER JOIN (
                        SELECT employee_id, COUNT(*) AS count
                        FROM assignments
                        GROUP BY employee_id
                        HAVING count >= 2
                    ) a ON e.id = a.employee_id
                    ORDER BY employee""")
for row in cursor.fetchall():
    print(f"  {row['employee']:<20} {row['count']} assignments")

conn.close()

--- Employees with Department (INNER JOIN) ---
  Alice Nguyen         Engineering
  Bob Martinez         Marketing
  Carol Kim            Engineering
  Dave Okafor          Sales
  Eve Patel            Marketing
  Frank Li             HR
  Grace Thompson       Engineering
  Hana Sato            Sales
  Ivan Rossi           Engineering
  Julia Fernandez      HR
  Kevin O'Brien        Sales
  Laura Chen           Marketing
  Marcus Reed          Engineering
  Nina Johansson       Engineering
  Omar Hassan          Sales
  Paula Diaz           HR
  Quinn Baker          Marketing
  Rachel Wong          Engineering
  Sam Nkosi            Sales
  Uma Sharma           Marketing
  Victor Alves         Engineering
  Wendy Park           Sales
  Xavier Dubois        HR
  Yuki Tanaka          Engineering
  Zoe Williams         Marketing

--- All Project Assignments ---
  Alice Nguyen         Apollo Platform           Tech Lead
  Alice Nguyen         Data Lake Migration       Architect
  Bob Marti

---
## Task 3 — LEFT JOIN Queries
**20 points**

LEFT JOIN returns **all rows from the left table**, with `NULL` on the right side when there is no match. This is how you find things that are missing or unassigned.

**Requirements:**
1. List **all 27 employees** with their department name. Employees with no department (`dept_id = NULL`) should show `"No Department"` in the output — use Python's `or` operator: `row["department"] or "No Department"`.
2. Find all **employees who have no project assignments** — use a LEFT JOIN between `employees` and `assignments`, then filter `WHERE a.id IS NULL`.
3. List **all departments** with their employee headcount, including `Legal` which has zero employees. Use `COUNT(e.id)` — not `COUNT(*)` — so the empty department shows `0`.
4. List **all projects** with their sponsoring department name. Projects with no department (`Skunkworks AI`) should show `"No Department"`.

> 💡 **Key rule:** `COUNT(*)` counts every row including NULL placeholders. `COUNT(e.id)` counts only non-NULL values — which gives the correct `0` for departments with no employees.

In [10]:
# Task 3 — LEFT JOIN
conn = sqlite3.connect("company_w4.db")
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

# 1. All 27 employees with department (NULL → "No Department")
print("--- All Employees with Department (LEFT JOIN) ---")
print(f"  {'Name':<22} {'Department':<18} {'City':<15} {'Salary':>12}")
print("  " + "-" * 70)
cursor.execute("""SELECT e.name AS employee, COALESCE(d.name, 'No Department') AS department,
                    e.city, e.salary
                    FROM employees e
                    LEFT JOIN departments d ON e.dept_id = d.id
                    ORDER BY employee""")
for row in cursor.fetchall():
    print(f"  {row['employee']:<22} {row['department']:<18} {row['city']:<15} ${row['salary']:>10,.2f}")

# 2. Employees with NO assignments
print("\n--- Employees with No Project Assignments ---")
cursor.execute("""SELECT e.name AS employee
                    FROM employees e
                    LEFT JOIN assignments a ON e.id = a.employee_id
                    WHERE a.id IS NULL
                    ORDER BY employee""")
for row in cursor.fetchall():
    print(f"  {row['employee']}")

# 3. Department headcount — must use COUNT(e.id) not COUNT(*)
print("\n--- Department Headcount (including empty depts) ---")
cursor.execute("""SELECT d.name AS department, COUNT(e.id) AS headcount
                    FROM departments d
                    LEFT JOIN employees e ON d.id = e.dept_id
                    GROUP BY d.id
                    ORDER BY department""")
for row in cursor.fetchall():
    print(f"  {row['department']:<18} {row['headcount']} employees")

# 4. All projects with sponsoring department
print("\n--- Projects with Department ---")
cursor.execute("""SELECT p.name AS project, COALESCE(d.name, 'No Department') AS department
                    FROM projects p
                    LEFT JOIN departments d ON p.dept_id = d.id
                    ORDER BY project""")
for row in cursor.fetchall():
    print(f"  {row['project']:<25} {row['department']}")

conn.close()

--- All Employees with Department (LEFT JOIN) ---
  Name                   Department         City                  Salary
  ----------------------------------------------------------------------
  Alice Nguyen           Engineering        San Francisco   $107,100.00
  Bob Martinez           Marketing          Chicago         $ 62,000.00
  Carol Kim              Engineering        San Francisco   $117,600.00
  Dave Okafor            Sales              New York        $ 71,000.00
  Eve Patel              Marketing          Chicago         $ 58,000.00
  Frank Li               HR                 New York        $ 84,000.00
  Grace Thompson         Engineering        Austin          $108,150.00
  Hana Sato              Sales              Austin          $ 47,000.00
  Ivan Rossi             Engineering        Austin          $102,900.00
  Julia Fernandez        HR                 San Francisco   $ 91,000.00
  Kevin O'Brien          Sales              New York        $ 67,500.00
  Laura Chen

---
## Task 4 — Multi-table Queries (3 and 4 tables)
**20 points**

Chain multiple JOINs to answer questions that require data from three or four tables at once.

**Requirements:**
1. For every assignment, show the **employee name**, their **department**, the **project name**, **project status**, their **role**, and **hours**. Use four tables: `assignments`, `employees`, `departments`, `projects`. Use LEFT JOIN for departments so contractors still appear.
2. Find all employees working on **Active** projects — show employee name, department, project name. Use a `WHERE p.status = 'Active'` after the joins. Deduplicate with `DISTINCT` so employees on multiple active projects appear only once.
3. Calculate the **total hours each department has logged** across all its employees' assignments. Show department name and total hours, ordered highest first. Include only departments that have logged at least 1 hour (`HAVING SUM(a.hours) > 0`).
4. Find the **single project** each employee has spent the most hours on. Show employee name, project name, and hours. Order by hours descending.

> 💡 **Hint for requirement 2:** Place the `WHERE` clause after all the `JOIN` clauses — SQL evaluates JOINs before WHERE filters.

In [11]:
# Task 4 — Multi-table queries
conn = sqlite3.connect("company_w4.db")
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

# 1. Full assignment report — 4 tables
print("--- Full Assignment Report ---")
print(f"  {'Employee':<22} {'Department':<14} {'Project':<26} {'Status':<11} {'Role':<14} {'Hrs':>4}")
print("  " + "-" * 98)
cursor.execute("""SELECT e.name AS employee, COALESCE(d.name, 'No Dept') AS department,
                    p.name AS project, p.status, a.role, a.hours
                    FROM assignments a
                    INNER JOIN employees e ON a.employee_id = e.id
                    LEFT JOIN departments d ON e.dept_id = d.id
                    INNER JOIN projects p ON a.project_id = p.id
                    ORDER BY employee""")
for row in cursor.fetchall():
    print(f"  {row['employee']:<22} {row['department']:<14} {row['project']:<26} {row['status']:<11} {row['role']:<14} {row['hours']:>4}")

# 2. Employees on Active projects (DISTINCT)
print("\n--- Employees on Active Projects ---")
cursor.execute("""SELECT DISTINCT e.name AS employee
                    FROM assignments a
                    INNER JOIN employees e ON a.employee_id = e.id
                    INNER JOIN projects p ON a.project_id = p.id
                    WHERE p.status = 'Active'
                    ORDER BY employee""")
for row in cursor.fetchall():
    print(f"  {row['employee']}")

# 3. Total hours logged per department
print("\n--- Total Hours Logged by Department ---")
cursor.execute("""SELECT COALESCE(d.name, 'No Dept') AS department, SUM(a.hours) AS total_hours
                    FROM assignments a
                    INNER JOIN employees e ON a.employee_id = e.id
                    LEFT JOIN departments d ON e.dept_id = d.id
                    GROUP BY d.id
                    ORDER BY department""")
for row in cursor.fetchall():
    print(f"  {row['department']:<18} {row['total_hours']} hours")

# 4. Each employee's highest-hours project
print("\n--- Each Employee's Top Project by Hours ---")
cursor.execute("""SELECT e.name AS employee, p.name AS project, a.hours
                    FROM assignments a
                    INNER JOIN employees e ON a.employee_id = e.id
                    INNER JOIN projects p ON a.project_id = p.id
                    WHERE (a.employee_id, a.hours) IN (
                        SELECT employee_id, MAX(hours)
                        FROM assignments
                        GROUP BY employee_id
                    )
                    ORDER BY employee""")
for row in cursor.fetchall():
    print(f"  {row['employee']:<20} {row['project']:<25} {row['hours']} hours")

conn.close()

--- Full Assignment Report ---
  Employee               Department     Project                    Status      Role            Hrs
  --------------------------------------------------------------------------------------------------
  Alice Nguyen           Engineering    Apollo Platform            Active      Tech Lead       180
  Alice Nguyen           Engineering    Data Lake Migration        Active      Architect       200
  Bob Martinez           Marketing      Brand Refresh              Completed   Lead            200
  Bob Martinez           Marketing      Q4 Campaign                On Hold     Manager         100
  Carol Kim              Engineering    Apollo Platform            Active      Architect       140
  Carol Kim              Engineering    Mobile App v2              Active      Tech Lead       160
  Dave Okafor            Sales          Sales Pipeline CRM         Active      Project Mgr      90
  Eve Patel              Marketing      Brand Refresh              Completed

---
## Task 5 — Error Handling for Database Operations
**15 points**

Wrap database operations in `try`/`except` so your program handles problems gracefully instead of crashing.

**Requirements:**
1. Write a function `get_employees_by_dept(dept_name)` that returns all employees in a given department. Wrap the database call in `try`/`except sqlite3.OperationalError`. Test it with `"Engineering"` and with `"Accounting"` (a dept that doesn't exist — it should return an empty list, not crash).
2. Write a function `add_project(name, dept_id, budget, status)` that inserts a new project. Wrap it in `try`/`except`. Test with a valid insert, then test again with the **same name** — the `name` column has a UNIQUE constraint, so the second insert should raise `sqlite3.IntegrityError`. Catch it and print a clear message.
3. Write a function `safe_divide(numerator, denominator)` that divides two numbers and returns the result. Use `try`/`except ZeroDivisionError` to return `0.0` instead of crashing when `denominator` is zero. Test it with `safe_divide(100, 4)` and `safe_divide(100, 0)`.

> 💡 **Hint for requirement 2:** After the first `add_project` call succeeds, verify the row exists with a SELECT, then call `add_project` again with the same name and watch the `IntegrityError` get caught.

In [12]:
# Task 5 — Error handling

# 1. get_employees_by_dept
def get_employees_by_dept(dept_name):
    """Return all employees in dept_name. Returns [] if none found."""
    try:
        conn = sqlite3.connect("company_w4.db")
        conn.row_factory = sqlite3.Row
        cursor = conn.cursor()
        cursor.execute("""SELECT e.name, e.salary
                            FROM employees e
                            INNER JOIN departments d ON e.dept_id = d.id
                            WHERE d.name = ?""", (dept_name,))
        return cursor.fetchall()

    except sqlite3.OperationalError as e:
        print(f"Database error: {e}")
        return []
    finally:
        conn.close()

# Test it
eng = get_employees_by_dept("Engineering")
print(f"Engineering: {len(eng)} employees")
for e in eng:
    print(f"  {e['name']:<22} ${e['salary']:>10,.2f}")

acct = get_employees_by_dept("Accounting")
print(f"\nAccounting: {len(acct)} employees  (expected 0 — dept doesn't exist)")


# 2. add_project with IntegrityError handling
def add_project(name, dept_id, budget, status):
    """Insert a new project. Catches IntegrityError on duplicate name."""
    try:
        conn = sqlite3.connect("company_w4.db")
        cursor = conn.cursor()
        cursor.execute("""INSERT INTO projects (name, dept_id, budget, status)
                            VALUES (?, ?, ?, ?)""", (name, dept_id, budget, status))
        conn.commit()
        print(f"Project '{name}' added successfully.")
    except sqlite3.IntegrityError as e:
        print(f"Integrity error: {e} — likely a duplicate project name.")
    except sqlite3.OperationalError as e:
        print(f"Database error: {e}")
    finally:
        conn.close()


# Test — first insert should succeed
print("\n--- add_project tests ---")
add_project("Test Initiative", 1, 50000.0, "Active")
# Second insert with same name should be caught
add_project("Test Initiative", 2, 75000.0, "On Hold")


# 3. safe_divide
def safe_divide(numerator, denominator):
    """Divide two numbers safely. Returns 0.0 if denominator is zero."""
    try:
        return numerator / denominator
    except ZeroDivisionError:
        print("Warning: Division by zero. Returning 0.0 instead.")
        return 0.0


print(f"\n100 / 4   = {safe_divide(100, 4)}")    # 25.0
print(f"100 / 0   = {safe_divide(100, 0)}")     # 0.0  (no crash)

Engineering: 9 employees
  Alice Nguyen           $107,100.00
  Carol Kim              $117,600.00
  Grace Thompson         $108,150.00
  Ivan Rossi             $102,900.00
  Marcus Reed            $124,950.00
  Nina Johansson         $103,950.00
  Rachel Wong            $112,350.00
  Victor Alves           $125,000.00
  Yuki Tanaka            $ 98,700.00

Accounting: 0 employees  (expected 0 — dept doesn't exist)

--- add_project tests ---
Project 'Test Initiative' added successfully.
Project 'Test Initiative' added successfully.

100 / 4   = 25.0
100 / 0   = 0.0


---
## Task 6 — Analytical Queries
**10 points**

Combine JOINs with aggregate functions to answer real business questions.

**Requirements:**
1. Find the **average salary per department**, including only departments that exist in both tables. Show department name and average salary, formatted with `$` and commas, ordered highest first.
2. Find the **most expensive active project** — show project name, department, and budget.
3. Find employees whose salary is **above their department's average**. Show name, department, salary, and the department average. Use a subquery:
```sql
WHERE e.salary > (
    SELECT AVG(e2.salary)
    FROM employees e2
    WHERE e2.dept_id = e.dept_id
)
```

> 💡 **Hint for requirement 3:** The subquery runs once per row in the outer query, comparing each employee's salary against the average of their own department.

In [13]:
# Task 6 — Analytical queries
conn = sqlite3.connect("company_w4.db")
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

# 1. Average salary by department
print("--- Average Salary by Department ---")
cursor.execute("""SELECT COALESCE(d.name, 'No Department') AS department, AVG(e.salary) AS avg_salary
                    FROM employees e
                    LEFT JOIN departments d ON e.dept_id = d.id
                    GROUP BY d.id
                    ORDER BY department""")
for row in cursor.fetchall():
    print(f"  {row['department']:<18} ${row['avg_salary']:>10,.2f}")

# 2. Most expensive active project
print("\n--- Most Expensive Active Project ---")
cursor.execute("""SELECT p.name AS project, p.budget
                    FROM projects p
                    WHERE p.status = 'Active'
                    ORDER BY p.budget DESC
                    LIMIT 1""")
for row in cursor.fetchall():
    print(f"  {row['project']:<25} ${row['budget']:>10,.2f}")

# 3. Employees above their department average
print("\n--- Employees Above Their Department Average ---")
print(f"  {'Name':<22} {'Department':<14} {'Salary':>12} {'Dept Avg':>12}")
print("  " + "-" * 64)
cursor.execute("""SELECT e.name AS employee, COALESCE(d.name, 'No Dept') AS department,
                    e.salary,
                    (SELECT AVG(e2.salary)
                    FROM employees e2
                        WHERE e2.dept_id = e.dept_id) AS dept_avg
                    FROM employees e
                    LEFT JOIN departments d ON e.dept_id = d.id
                    WHERE e.salary > (
                        SELECT AVG(e2.salary)
                        FROM employees e2
                        WHERE e2.dept_id = e.dept_id
                    )
                    ORDER BY employee""")
for row in cursor.fetchall():
    print(f"  {row['employee']:<22} {row['department']:<14} ${row['salary']:>10,.2f} ${row['dept_avg']:>10,.2f}")
conn.close()

--- Average Salary by Department ---
  Engineering        $111,188.89
  HR                 $ 86,125.00
  Marketing          $ 61,833.33
  No Department      $ 70,000.00
  Sales              $ 57,500.00

--- Most Expensive Active Project ---
  Apollo Platform           $920,000.00

--- Employees Above Their Department Average ---
  Name                   Department           Salary     Dept Avg
  ----------------------------------------------------------------
  Bob Martinez           Marketing      $ 62,000.00 $ 61,833.33
  Carol Kim              Engineering    $117,600.00 $111,188.89
  Dave Okafor            Sales          $ 71,000.00 $ 57,500.00
  Julia Fernandez        HR             $ 91,000.00 $ 86,125.00
  Kevin O'Brien          Sales          $ 67,500.00 $ 57,500.00
  Marcus Reed            Engineering    $124,950.00 $111,188.89
  Paula Diaz             HR             $ 87,500.00 $ 86,125.00
  Quinn Baker            Marketing      $ 66,000.00 $ 61,833.33
  Rachel Wong           

---
## Task 7 — Debugging Exercise
**20 points**

Open `debug_me.py` in VS Code and fix all 7 bugs using the debugger. Refer to `week4_debugging_procedure.md` for step-by-step guidance on each bug.

**The 7 bugs:**

| # | Function | Bug type |
|---|----------|----------|
| 1 | `get_all_departments` | `NameError` — wrong variable name |
| 2 | `get_all_employees_with_dept` | Wrong JOIN type — `INNER` should be `LEFT` |
| 3 | `get_projects_by_status` | Missing trailing comma in tuple |
| 4 | `get_department_headcount` | `COUNT(*)` should be `COUNT(e.id)` |
| 5 | `get_employee_project_summary` | `KeyError` — column not in SELECT |
| 6 | `update_project_status` | Missing `conn.commit()` |
| 7 | `total_hours_per_project` | Infinite loop — `i` never increments |

**Requirements:**
1. Fix all 7 bugs in `debug_me.py`.
2. Run the fixed script from Terminal: `python3 debug_me.py`
3. Confirm all 7 tests pass with no errors and no infinite loops.
4. In the cell below, write a short note (1–2 sentences each) explaining what each bug was and how you fixed it.

> ⚠️ **Debugging tips:**
> - Work through bugs **in order** — the program crashes on Bug 1 so you won't reach later bugs until it's fixed
> - Use **F10** (Step Over) to watch code run line by line
> - Use **F11** (Step Into) to follow execution inside a function
> - Watch the **Variables panel** — it shows you what values actually are, not what you assumed they'd be
> - For Bug 7 (infinite loop), press **Shift+F5** immediately to stop — do not wait for it to finish

### Your debugging notes

Fill in each section after fixing the bug:

**Bug 1 — NameError:**  
*(what was wrong and how you fixed it)*

**Bug 2 — Wrong JOIN:**  

**Bug 3 — Tuple comma:**  

**Bug 4 — COUNT:**  

**Bug 5 — KeyError:**  

**Bug 6 — Missing commit:**  

**Bug 7 — Infinite loop:**  

---
## Task 8 — Commit to GitHub
**10 points**

Save and push all your Week 4 files.

**Steps:**
1. Save this notebook: **File → Save Notebook** (`Cmd+S`).
2. Make sure these files are in your `week4/` folder:
   - `week4_assignment.ipynb`
   - `company_w4.db`
   - `debug_me.py` (with all bugs fixed)
3. Open Terminal and run:

```bash
cd ~/Documents/python-course
git add .
git commit -m "Week 4: joins, error handling, and debugging exercise"
git push
```

---
## Grading Rubric

| Task | Points | Full marks when… |
|------|--------|-------------------|
| 1 — Schema | 10 | All 4 tables listed with columns and counts; FileNotFoundError handled |
| 2 — INNER JOIN | 15 | All 4 queries correct; HAVING used for requirement 4 |
| 3 — LEFT JOIN | 20 | All 27 employees shown; empty depts show 0; NULL shown as "No Department" |
| 4 — Multi-table | 20 | 4-table join correct; DISTINCT used; subquery-style logic where needed |
| 5 — Error handling | 15 | All three functions handle errors without crashing |
| 6 — Analytics | 10 | Subquery used; results formatted correctly |
| 7 — Debugging | 20 | All 7 bugs fixed; notes written for each bug |
| **Total** | **100 (+ 10 for GitHub)** | |

---
## Before You Submit — Checklist

- [ ] All notebook cells run without errors (**Kernel → Restart & Run All**)
- [ ] Task 3 requirement 1 returns exactly **27 rows**
- [ ] Task 3 requirement 3 shows `Legal` with **0 employees**
- [ ] Task 5 catches `IntegrityError` on duplicate project insert
- [ ] `debug_me.py` runs completely with all 7 tests passing
- [ ] Debugging notes filled in for all 7 bugs
- [ ] All files committed to `week4/` on GitHub

In [14]:
# Self-check — DO NOT EDIT THIS CELL
import sqlite3

print("=" * 50)
print("  Week 4 Self-check")
print("=" * 50)

checks = []

conn = sqlite3.connect("company_w4.db")
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

# Check 1: all four tables exist
try:
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = {r["name"] for r in cursor.fetchall()}
    expected = {"departments", "employees", "projects", "assignments"}
    assert expected.issubset(tables), f"Missing tables: {expected - tables}"
    checks.append(("All 4 tables present", True, ""))
except Exception as e:
    checks.append(("All 4 tables present", False, str(e)))

# Check 2: LEFT JOIN returns 27 employees
try:
    cursor.execute("""
        SELECT COUNT(*) AS n FROM employees e
        LEFT JOIN departments d ON d.id = e.dept_id
    """)
    n = cursor.fetchone()["n"]
    assert n == 27, f"Expected 27, got {n}"
    checks.append(("LEFT JOIN returns all 27 employees", True, ""))
except Exception as e:
    checks.append(("LEFT JOIN returns all 27 employees", False, str(e)))

# Check 3: INNER JOIN returns only 25
try:
    cursor.execute("""
        SELECT COUNT(*) AS n FROM employees e
        INNER JOIN departments d ON d.id = e.dept_id
    """)
    n = cursor.fetchone()["n"]
    assert n == 25, f"Expected 25, got {n}"
    checks.append(("INNER JOIN returns 25 (excludes NULL dept)", True, ""))
except Exception as e:
    checks.append(("INNER JOIN returns 25 (excludes NULL dept)", False, str(e)))

# Check 4: Legal dept shows 0 with COUNT(e.id)
try:
    cursor.execute("""
        SELECT d.name, COUNT(e.id) AS headcount
        FROM departments d
        LEFT JOIN employees e ON e.dept_id = d.id
        GROUP BY d.id
    """)
    rows = cursor.fetchall()
    legal = [r for r in rows if r["name"] == "Legal"]
    assert legal, "Legal department not found"
    assert legal[0]["headcount"] == 0, f"Legal headcount should be 0, got {legal[0]['headcount']}"
    checks.append(("Legal dept headcount = 0 with COUNT(e.id)", True, ""))
except Exception as e:
    checks.append(("Legal dept headcount = 0 with COUNT(e.id)", False, str(e)))

# Check 5: 3 employees have no assignments
try:
    cursor.execute("""
        SELECT COUNT(*) AS n FROM employees e
        LEFT JOIN assignments a ON a.employee_id = e.id
        WHERE a.id IS NULL
    """)
    n = cursor.fetchone()["n"]
    assert n == 3, f"Expected 3 unassigned employees, got {n}"
    checks.append(("3 employees have no assignments", True, ""))
except Exception as e:
    checks.append(("3 employees have no assignments", False, str(e)))

# Check 6: safe_divide function exists and handles zero
try:
    assert callable(safe_divide), "safe_divide is not defined"
    assert safe_divide(100, 4) == 25.0, "safe_divide(100,4) should return 25.0"
    assert safe_divide(100, 0) == 0.0, "safe_divide(100,0) should return 0.0"
    checks.append(("safe_divide handles zero correctly", True, ""))
except Exception as e:
    checks.append(("safe_divide handles zero correctly", False, str(e)))

conn.close()

# Print results
passed = 0
for name, ok, msg in checks:
    icon   = "✓" if ok else "✗"
    status = "PASS" if ok else "FAIL"
    print(f"  {icon} {name}: {status}")
    if msg:
        print(f"      → {msg}")
    if ok:
        passed += 1

print("=" * 50)
print(f"  {passed}/{len(checks)} checks passed")
if passed == len(checks):
    print("  All good — ready to push to GitHub!")
else:
    print("  Fix the issues above and run again.")
print("=" * 50)

  Week 4 Self-check
  ✓ All 4 tables present: PASS
  ✓ LEFT JOIN returns all 27 employees: PASS
  ✓ INNER JOIN returns 25 (excludes NULL dept): PASS
  ✓ Legal dept headcount = 0 with COUNT(e.id): PASS
  ✓ 3 employees have no assignments: PASS
  ✓ safe_divide handles zero correctly: PASS
  6/6 checks passed
  All good — ready to push to GitHub!
